# Fine-Tuning

In this lab you will explore fine-tuning a CNN to classify pet breeds.

In [ ]:
import keras
import numpy as np
from matplotlib import pyplot as plt

In this lab we will use a prepared version of the [Oxford-IIIT Pet dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/).

In [ ]:
import os
if not os.path.exists('oxford_pets.zip'):
  !wget -q "https://www.dropbox.com/scl/fi/p49ifha27c2u3uptfj42w/oxford_pets_corrected.zip?rlkey=dwk3dsptzir8v846imsq6bgw3&dl=1" -O oxford_pets.zip
  !unzip -qq oxford_pets.zip

We will use the Keras function `image_dataset_from_directory` to load the images.  It expects the images to be stored in separate directories according to their labels:

```
   dog/
       - dog1.jpg
       - dog2.jpg
       - ...
   cat/
       - cat1.jpg
       - cat2.jpg
       - ...
```

It returns a Tensorflow `Dataset` object.  Note that it does not load the images from disk -- it just looks through the directory and catalogs which images are available.

In [ ]:
train_ds = keras.preprocessing.image_dataset_from_directory('oxford_pets/train')
train_ds

In [ ]:
train_ds.class_names

When we iterate over the dataset, it loads batches of images from disk.  The batch size is set by the `batch_size` argument to `image_dataset_from_directory`.

Here `.take(1)` tells the dataset we only want the first batch.

Because the data is returned as `EagerTensor`s, we have to call `.numpy()` for them to be actually loaded and converted to Numpy arrays.

In [ ]:
for images, labels in train_ds.take(1):
  print('images:',images.shape,images.dtype,'labels:',labels.shape,labels.dtype)
  print('image data range:',images[0].numpy().min(),images[0].numpy().max())
  plt.imshow(images[0].numpy().astype('uint8'))
  plt.title(train_ds.class_names[labels[0].numpy()])
  plt.show()

`image_dataset_from_directory` resizes the images so that they all have the same shape.  You can control the image size through the `image_size` argument.  The default is $256\times256$.

If the original image is not square, then the image will be somewhat squashed by the resize operation.  To avoid this, you can set `crop_to_aspect_ratio=True` so that it will center crop the image before resizing.

`image_dataset_from_directory` can automatically create a validation split for you, using the `validation_split` argument.  You need to call the function twice: once with `subset='train'` and once with `subset='validation'` to make both datasets.  And, you should set the `seed` argument to ensure that the same split is used both times!

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    'oxford_pets/train',
    image_size=(224,224),
    subset='training',
    validation_split=0.1,
    seed=42
)
val_ds = keras.utils.image_dataset_from_directory(
    'oxford_pets/train',
    image_size=(224,224),
    subset='validation',
    validation_split=0.1,
    seed=42,
)
test_ds = keras.utils.image_dataset_from_directory(
    'oxford_pets/test',
    image_size=(224,224)
)

### Exercises

1. Create a `Sequential` model with the pre-trained `VGG16` network.

The model should have the following layers:
- Input layer
- One or more [data augmentation](https://keras.io/api/layers/preprocessing_layers/image_augmentation/) layers such as `RandomFlip`, `RandomZoom`, `RandomRotation`, etc.
- VGG16 preprocess function inside a `Lambda` layer
- VGG16 layer: don't include top; use `max` pooling.
- Dense layer with 35 outputs and Softmax activation function for multi-class classification

Set the VGG16 part of the network to be fixed by setting the `trainable` attribute of VGG16 layer to `False`.  (You can access the layers of the model with `model.layers`.)

Check the model summary to make sure that the VGG16 layer is trainable (it should report a very large number of non-trainable parameters, and a small number of trainable parameters.)


Compile the model with sparse categorical cross entropy loss and accuracy metric.  Use Adam optimizer with learning rate 3e-4.

4. Evaluate the model on the test set and check that the accuracy is about $1/35=.029$.

5. Now train the model on the training set (don't forget to include the validation set) for 20 epochs.

6. Evaluate the accuracy of the fine-tuned model on the test set.

7. The following code will show some test images with the correct and predicted labels.

In [ ]:
images, labels = next(iter(test_ds))
preds = model.predict(images)

In [ ]:
for im,label,pred in zip(images,labels,preds):
    plt.imshow(im.numpy().astype('uint8'))
    plt.title(f'correct: {test_ds.class_names[label]} | predicted: {test_ds.class_names[np.argmax(pred)]}')
    plt.show()